<a href="https://colab.research.google.com/github/sreevanimtcs2502/sreevanimtcs2502/blob/nlp/NLPE_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clean install correct libraries
!pip install -q --upgrade --force-reinstall transformers==4.37.2 sentencepiece gradio
!pip install -q git+https://github.com/openai/whisper.git

import whisper
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

print("Loading models...")

# Speech Recognition
speech_model = whisper.load_model("base")

# Telugu → English Translation
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
translator = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Summarization model
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

print("Models loaded successfully")

def process(audio):

    # Speech → Telugu
    result = speech_model.transcribe(audio, language="te")
    telugu_text = result["text"]

    # Telugu → English
    inputs = tokenizer(telugu_text, return_tensors="pt")
    tokens = translator.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id["eng_Latn"]
    )
    english_text = tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]

    # English → Summary
    summary = summarizer(english_text, max_length=60, min_length=15)[0]["summary_text"]

    return telugu_text, english_text, summary


app = gr.Interface(
    fn=process,
    inputs=gr.Audio(sources=["microphone"], type="filepath"),
    outputs=[
        gr.Textbox(label="Telugu Text"),
        gr.Textbox(label="English Translation"),
        gr.Textbox(label="Summary")
    ],
    title="Telugu Speech → English Summary"
)

app.launch()

In [ ]:
# Install libraries
!pip install -q --upgrade --force-reinstall transformers==4.37.2 sentencepiece gradio
!pip install -q git+https://github.com/openai/whisper.git

import whisper
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

print("Loading models...")

# Speech Recognition Model
speech_model = whisper.load_model("base")

# Telugu → English Translation Model
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
translator = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Summarization Model
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

print("Models loaded successfully")

def process(audio_file):

    # Speech → Telugu Text
    result = speech_model.transcribe(audio_file, language="te")
    telugu_text = result["text"]

    # Telugu → English Translation
    inputs = tokenizer(telugu_text, return_tensors="pt")
    tokens = translator.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id["eng_Latn"]
    )
    english_text = tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]

    # English → Summary
    summary = summarizer(
        english_text,
        max_length=60,
        min_length=15
    )[0]["summary_text"]

    return telugu_text, english_text, summary


app = gr.Interface(
    fn=process,
    inputs=gr.Audio(type="filepath", label="Upload Telugu Audio File"),
    outputs=[
        gr.Textbox(label="Telugu Text"),
        gr.Textbox(label="English Translation"),
        gr.Textbox(label="Summary")
    ],
    title="Telugu Speech → English Translation and Summary"
)

app.launch()

In [ ]:
# Install libraries
!pip uninstall -y gradio transformers sentence-transformers -q
!pip install -q --upgrade --force-reinstall gradio==4.44.1 transformers==4.37.2 sentencepiece huggingface_hub>=0.19.3

import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading models...")

# Telugu → English translation model
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
translator = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully")

def summarize_text(text):
    sentences = text.split(".")
    return ".".join(sentences[:2]) if len(sentences) > 2 else text


def process(telugu_text):

    # Telugu → English translation
    inputs = tokenizer(telugu_text, return_tensors="pt")
    tokens = translator.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id["eng_Latn"],
        max_length=512
    )

    english_text = tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]

    # Summarization
    summary = summarize_text(english_text)

    return english_text, summary


app = gr.Interface(
    fn=process,
    inputs=gr.Textbox(lines=5, label="Enter Telugu Text"),
    outputs=[
        gr.Textbox(label="English Translation"),
        gr.Textbox(label="Summary")
    ],
    title="Telugu Text → English Translation → Summary"
)

app.launch(share=True)

In [1]:
!pip install gTTS

from gtts import gTTS
from google.colab import files
from IPython.display import Audio

# Long Telugu paragraph (good for demo + summarization)
telugu_text = """
ఈ రోజు మనం కృత్రిమ మేధస్సు మరియు యంత్ర అధ్యయనం గురించి మాట్లాడబోతున్నాం.
కృత్రిమ మేధస్సు అనేది కంప్యూటర్లకు మనుషులలా ఆలోచించే మరియు నిర్ణయాలు తీసుకునే సామర్థ్యాన్ని అందించే సాంకేతికత.
ఇది వైద్యం, విద్య, రవాణా, మరియు వ్యాపారం వంటి అనేక రంగాలలో ఉపయోగపడుతోంది.

ఉదాహరణకు, వైద్య రంగంలో కృత్రిమ మేధస్సు వ్యాధులను త్వరగా గుర్తించడంలో సహాయపడుతుంది.
విద్యా రంగంలో ఇది విద్యార్థులకు వ్యక్తిగతంగా నేర్చుకునే అవకాశాన్ని ఇస్తుంది.
ఇక రవాణా రంగంలో స్వయం నడిచే వాహనాల అభివృద్ధికి ఇది సహాయపడుతోంది.

భవిష్యత్తులో కృత్రిమ మేధస్సు మన జీవన విధానాన్ని మరింత సులభతరం చేస్తుంది.
కానీ దీనిని బాధ్యతతో మరియు జాగ్రత్తగా ఉపయోగించడం చాలా అవసరం.
అందుకే శాస్త్రవేత్తలు మరియు పరిశోధకులు ఈ సాంకేతికతను సమాజానికి ఉపయోగపడే విధంగా అభివృద్ధి చేస్తున్నారు.
"""

# Generate audio
tts = gTTS(text=telugu_text, lang="te")

file_name = "telugu_presentation_audio.mp3"
tts.save(file_name)

print("Audio generated successfully!")

# Play audio in Colab
Audio(file_name)

# Download audio
files.download(file_name)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
Audio generated successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# Install libraries (run once in Colab/Jupyter)
!pip install openai-whisper transformers googletrans==4.0.0-rc1 torch

import whisper
from transformers import pipeline
from googletrans import Translator

# Load speech recognition model
model = whisper.load_model("base")

# Step 1: Speech to Text
result = model.transcribe("/content/nature_speech_voice.wav")
english_text = result["text"]

# Step 2: Translate English → Telugu
translator = Translator()
telugu_translation = translator.translate(english_text, src='en', dest='te').text

# Step 3: Summarization
summarizer = pipeline("summarization")
summary = summarizer(english_text, max_length=60, min_length=25, do_sample=False)[0]['summary_text']

# Output
print("\n--- Transcribed English Text ---\n")
print(english_text)

print("\n--- Telugu Translation ---\n")
print(telugu_translation)

print("\n--- Summary ---\n")
print(summary)

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0



--- Transcribed English Text ---

 Hello everyone! Today I would like to talk about the importance and beauty of nature. Nature is the foundation of life on earth. It includes everything around us, such as forests, rivers, mountains, oceans, plants, animals, and even there we breathe. Every element in nature works together in a balanced system called an ecosystem. This balance allows living organisms to survive and grow. Forests play a very important role in maintaining this balance, trees produce oxygen and absorb carbon dioxide, which helps regulate the climate. Forests are also home to many species of animals, birds, and insects. The organisms depend on plants and trees for food, shelter, and protection. Without forests, many living species would struggle to survive. Rivers and lakes are another essential part of nature. They provide fresh water at humans, animals, and plants in every day. Farmers depend on rivers to irrigate crops, and communities rely on clean water for drinking 